In [69]:
include("CRD_STA.jl")
using LinearAlgebra
using NonlinearEigenproblems
using DelimitedFiles
using ProgressMeter
using Plots

In [4]:
function BF(N_cheb,Ro,Tw,Mr)
    gamma = 1.4
    sigma = 0.72
    Co = 2 - Ro - Ro^2
    u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
    H,T = T_ca(Mr,f,q,w0,gamma,Tw)
    F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
    lam = - (2/3) * T
    kappa = (1/sigma) * T
    return F,G,H,T,rho,x,lam,kappa,D,D2
end
function eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num)
    A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2)
    nep = PEP([A0,A1,A2]); 
    eigval,eigvec = iar(nep,σ = c , neigs = num ,maxit = 500,tol=1e-10)
    # eigval = conj(eigval)
    return eigval,eigvec
end

eigsol (generic function with 1 method)

In [71]:
N_cheb = 99
Ro = -1
Tw = 1
Mr = 0.1
gamma = 1.4
sigma = 0.72
Co = 2-Ro-Ro^2
be = 0.0776
num = 1
omega = 0
F,G,H,T,rho,z,lam,kappa,D,D2 = BF(N_cheb,Ro,Tw,Mr)
R = 285.36
Ma = Mr/R

┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/LFeTa/src/integrator_interface.jl:665
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/LFeTa/src/integrator_interface.jl:665
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/LFeTa/src/integrator_interface.jl:665
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/LFeTa/src/integrator_interface.jl:665
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/LFeTa/src/integrator_interface.jl:665
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/LFeTa/src/integrator_interface.jl:665
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/LFeTa/src/integrator_interface.jl:665
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBas

0.0003504345388281469

In [73]:
eigval1,eigvec1 = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,0.3,num)
eigval1

1-element Vector{ComplexF64}:
 0.38634138120805184 + 0.0005468131528416272im

In [92]:
function cof_mat(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2)
    eye = I(N_cheb + 1)
    Zero = zeros(N_cheb + 1,N_cheb + 1)
    if Ro == -1 
        Ro = 1
    end
    Ta_11 = Ta_12 = Ta_13 = Ta_15 = Ta_22 = Ta_23 = Ta_24 = Ta_25 = Ta_31 = Ta_33 = Ta_34 = Ta_35 = Ta_41 = Ta_42 = Ta_44 = Ta_45 = Ta_51 = Ta_52 = Ta_53 = Zero
    Ta_14 = R * eye
    Ta_21 = R .* rho .* eye
    Ta_32 = R .* rho .* eye
    Ta_43 = R .* rho .* eye
    Ta_54 = (1-gamma)/(gamma) * R .*T .* eye
    Ta_55 = 1/(gamma) * R .*rho .* eye

    A_12 = A_13 = A_15 = A_22 = A_31 = A_33 = A_34 = A_35 = A_42 = A_44 = A_51 = A_52 = Zero
    A_11 = R * rho .* eye
    A_14 = R * F .* eye
    A_21 = R * rho .* F .* eye
    A_23 = - rho .* (D*T) .* eye
    A_24 = (gamma*Ma^2)^(-1) * R .* T .* eye
    A_25 = (gamma*Ma^2)^(-1) * R .* rho .* eye
    A_32 = R * rho .* F .* eye
    A_41 = -rho .* (D*lam) .* eye
    A_43 = R * rho .* F .* eye
    A_45 = - rho .* (D*F) .* eye
    A_53 = -2 * (gamma-1) * Ma^2 * (D*F) .* eye 
    A_54 = -(gamma-1)/(gamma) * R .* T .* F .* eye
    A_55 = 1/gamma * R * rho .* F .* eye

    B_11 = B_13 = B_15 = B_22 = B_23 = B_24 = B_25 = B_31 = B_41 = B_44 = B_51 = B_52 = Zero
    B_12 = R* rho .* eye
    B_14 = R * G .* eye
    B_21 = R * rho .* G .* eye
    B_32 = R * rho .* G .* eye
    B_33 = -rho .* (D*T) .* eye
    B_34 = (gamma * Ma^2)^(-1)  * R .* T .* eye
    B_35 = (gamma * Ma^2)^(-1)  * R .* rho .* eye
    B_42 = -rho .* (D*lam) .* eye
    B_43 = R * rho .* G .* eye
    B_45 = -rho .* (D*G) .* eye
    B_53 = -2 * (gamma-1) * Ma^2 * (D*G) .* eye
    B_54 = -(gamma-1)/(gamma) * R * T .* G .* eye
    B_55 = 1/gamma * R .* rho .* G .* eye

    C_11 = C_12 = C_15 = C_22 = C_23 = C_24 = C_31 = C_33 = C_34 = C_41 = C_42 = C_53 = Zero
    C_13 = R * rho.^2 .* eye
    C_14 = rho .* H .* eye
    C_21 = Ro * rho.^2  .* H .* eye
    C_25 = -rho.^2 .* (D*F) .* eye
    C_32 = Ro * rho.^2 .* H .* eye
    C_35 = -rho.^2 .* (D*G) .* eye
    C_43 = Ro * rho.^2 .* H .* eye
    C_44 = R * (gamma * Ma^2)^(-1) * rho .* T .* eye
    C_45 = R * (gamma * Ma^2)^(-1) * rho .* rho .* eye
    C_51 = -2 * (gamma-1) * Ma^2 * rho .* (D*F) .* eye
    C_52 = -2 * (gamma-1) * Ma^2 * rho .* (D*G) .* eye
    C_54 = -(gamma-1)/gamma * rho .* H .* T .* eye
    C_55 = 1/gamma * rho.^2 .* H .* eye + 1/sigma * rho.^2 .* (D*T) .* eye

    D_12 = D_15 = D_41 = D_42 = D_51 = D_52 = Zero
    D_11 = rho .* eye
    D_13 = R * rho .* (D*rho) .* eye
    D_14 = 2 * F .* eye + rho .* (D*H) .* eye
    D_21 = Ro * rho .* F .* eye
    D_22 = -rho .* (2* Ro * G .+ Co) .* eye
    D_23 = R * rho.^2 .* (D*F) .* eye
    D_24 = D*rho .* (D2*F) .* eye
    D_25 = -rho .* D*(rho.*(D*F)) .* eye
    D_31 = rho .* (2* Ro * G .+ Co) .* eye
    D_32 = Ro * rho .* F .* eye
    D_33 = R * rho.^2 .* (D*G) .* eye
    D_34 = F .* (2* Ro * G .+ Co) .* eye + Ro * rho .* H .* (D*G) .* eye
    D_35 = -rho.* (D * (rho .* (D*G))) .* eye
    D_43 = Ro * rho.^2 .* (D*H) .* eye
    D_44 = R * (gamma * Ma^2)^(-1) * rho .* (D*T) .* eye
    D_45 = R * (gamma * Ma^2)^(-1) * rho .* (D*rho) .* eye
    D_53 = R * rho.^2 .* (D*T) .* eye
    D_54 = 1/gamma * rho .* H .* (D*T) .* eye
    D_55 = -(gamma-1)/gamma * rho .* H .* (D*rho) .* eye - (1/sigma) * (rho .* (D*rho) .* (D*T) .+ rho.^2 .* (D2 * T)) .* eye - (gamma-1) * Ma^2 * rho.^2 .* ((D*F).^2 + (D*G).^2) .* eye

    Vxx_11 = Vxx_12 = Vxx_13 = Vxx_14 = Vxx_15 = Vxx_22 = Vxx_23 = Vxx_24 = Vxx_25 = Vxx_31 = Vxx_33 = Vxx_34 = Vxx_35 = Vxx_41 = Vxx_42 = Vxx_44 = Vxx_45 = Vxx_51 = Vxx_52 = Vxx_53 = Vxx_54 = Zero
    Vxx_21 = -(lam + 2*T) .* eye
    Vxx_32 = -T .* eye
    Vxx_43 = -T .* eye
    Vxx_55 = -kappa .* eye

    Vyy_11 = Vyy_12 = Vyy_13 = Vyy_14 = Vyy_15 = Vyy_22 = Vyy_23 = Vyy_24 = Vyy_25 = Vyy_31 = Vyy_33 = Vyy_34 = Vyy_35 = Vyy_41 = Vyy_42 = Vyy_44 = Vyy_45 = Vyy_51 = Vyy_52 = Vyy_53 = Vyy_54 = Zero
    Vyy_21 = -T .* eye
    Vyy_32 = -(lam .+ 2*T) .* eye
    Vyy_43 = -T .* eye
    Vyy_55 = -kappa .* eye

    Vzz_11 = Vzz_12 = Vzz_13 = Vzz_14 = Vzz_15 = Vzz_22 = Vzz_23 = Vzz_24 = Vzz_25 = Vzz_31 = Vzz_33 = Vzz_34 = Vzz_35 = Vzz_41 = Vzz_42 = Vzz_44 = Vzz_45 = Vzz_51 = Vzz_52 = Vzz_53 = Vzz_54 = Zero
    Vzz_21 = -rho .* eye
    Vzz_32 = -rho .* eye
    Vzz_43 = -rho .* (2 .+ lam .* rho) .* eye
    Vzz_55 = -rho.^2 .* kappa .* eye

    Vxy_11 = Vxy_12 = Vxy_13 = Vxy_14 = Vxy_15 = Vxy_21 = Vxy_23 = Vxy_24 = Vxy_25 = Vxy_32 = Vxy_33 = Vxy_34 = Vxy_35 = Vxy_41 = Vxy_42  = Vxy_43 = Vxy_44 = Vxy_45 = Vxy_51 = Vxy_52 = Vxy_53 = Vxy_54 = Vxy_55 =  Zero
    Vxy_22 = -(lam .+ T) .* eye
    Vxy_31 = -(lam .+ T) .* eye

    Vxz_11 = Vxz_12 = Vxz_13 = Vxz_14 = Vxz_15 = Vxz_21 = Vxz_22 = Vxz_24 = Vxz_25 = Vxz_31 = Vxz_32 = Vxz_33 = Vxz_34 = Vxz_35 = Vxz_42 = Vxz_43 = Vxz_44 = Vxz_45 = Vxz_51 = Vxz_52 = Vxz_53 = Vxz_54 = Vxz_55 =  Zero
    Vxz_23 = - (1 .+ rho.*lam) .* eye
    Vxz_41 = - (1 .+ rho.*lam) .* eye

    Vyz_11 = Vyz_12 = Vyz_13 = Vyz_14 = Vyz_15 = Vyz_21 = Vyz_22 = Vyz_23 = Vyz_24 = Vyz_25 = Vyz_31 = Vyz_32  = Vyz_34 = Vyz_35 = Vyz_41 = Vyz_43 = Vyz_44 = Vyz_45 = Vyz_51 = Vyz_52 = Vyz_53 = Vyz_54 = Vyz_55 =  Zero
    Vyz_33 = - (1 .+ rho.*lam) .* eye
    Vyz_42 = - (1 .+ rho.*lam) .* eye
    
    Ta = [Ta_11 Ta_12 Ta_13 Ta_14 Ta_15;Ta_21 Ta_22 Ta_23 Ta_24 Ta_25;Ta_31 Ta_32 Ta_33 Ta_34 Ta_35;Ta_41 Ta_42 Ta_43 Ta_44 Ta_45;Ta_51 Ta_52 Ta_53 Ta_54 Ta_55]

    A = [A_11 A_12 A_13 A_14 A_15;A_21 A_22 A_23 A_24 A_25;A_31 A_32 A_33 A_34 A_35;A_41 A_42 A_43 A_44 A_45;A_51 A_52 A_53 A_54 A_55]

    B = [B_11 B_12 B_13 B_14 B_15;B_21 B_22 B_23 B_24 B_25;B_31 B_32 B_33 B_34 B_35;B_41 B_42 B_43 B_44 B_45;B_51 B_52 B_53 B_54 B_55]

    C = [C_11 C_12 C_13 C_14 C_15;C_21 C_22 C_23 C_24 C_25;C_31 C_32 C_33 C_34 C_35;C_41 C_42 C_43 C_44 C_45;C_51 C_52 C_53 C_54 C_55]

    D1 = [D_11 D_12 D_13 D_14 D_15;D_21 D_22 D_23 D_24 D_25;D_31 D_32 D_33 D_34 D_35;D_41 D_42 D_43 D_44 D_45;D_51 D_52 D_53 D_54 D_55]

    Vxx = [Vxx_11 Vxx_12 Vxx_13 Vxx_14 Vxx_15;Vxx_21 Vxx_22 Vxx_23 Vxx_24 Vxx_25;Vxx_31 Vxx_32 Vxx_33 Vxx_34 Vxx_35;Vxx_41 Vxx_42 Vxx_43 Vxx_44 Vxx_45;Vxx_51 Vxx_52 Vxx_53 Vxx_54 Vxx_55]

    Vyy = [Vyy_11 Vyy_12 Vyy_13 Vyy_14 Vyy_15;Vyy_21 Vyy_22 Vyy_23 Vyy_24 Vyy_25;Vyy_31 Vyy_32 Vyy_33 Vyy_34 Vyy_35;Vyy_41 Vyy_42 Vyy_43 Vyy_44 Vyy_45;Vyy_51 Vyy_52 Vyy_53 Vyy_54 Vyy_55]

    Vzz = [Vzz_11 Vzz_12 Vzz_13 Vzz_14 Vzz_15;Vzz_21 Vzz_22 Vzz_23 Vzz_24 Vzz_25;Vzz_31 Vzz_32 Vzz_33 Vzz_34 Vzz_35;Vzz_41 Vzz_42 Vzz_43 Vzz_44 Vzz_45;Vzz_51 Vzz_52 Vzz_53 Vzz_54 Vzz_55]

    Vxy = [Vxy_11 Vxy_12 Vxy_13 Vxy_14 Vxy_15;Vxy_21 Vxy_22 Vxy_23 Vxy_24 Vxy_25;Vxy_31 Vxy_32 Vxy_33 Vxy_34 Vxy_35;Vxy_41 Vxy_42 Vxy_43 Vxy_44 Vxy_45;Vxy_51 Vxy_52 Vxy_53 Vxy_54 Vxy_55]

    Vxz = [Vxz_11 Vxz_12 Vxz_13 Vxz_14 Vxz_15;Vxz_21 Vxz_22 Vxz_23 Vxz_24 Vxz_25;Vxz_31 Vxz_32 Vxz_33 Vxz_34 Vxz_35;Vxz_41 Vxz_42 Vxz_43 Vxz_44 Vxz_45;Vxz_51 Vxz_52 Vxz_53 Vxz_54 Vxz_55]

    Vyz = [Vyz_11 Vyz_12 Vyz_13 Vyz_14 Vyz_15;Vyz_21 Vyz_22 Vyz_23 Vyz_24 Vyz_25;Vyz_31 Vyz_32 Vyz_33 Vyz_34 Vyz_35;Vyz_41 Vyz_42 Vyz_43 Vyz_44 Vyz_45;Vyz_51 Vyz_52 Vyz_53 Vyz_54 Vyz_55]
    D_1 = [D D D D D ; D D D D D; D D D D D; D D D D D; D D D D D]
    D2_1 = [D2 D2 D2 D2 D2;D2 D2 D2 D2 D2;D2 D2 D2 D2 D2;D2 D2 D2 D2 D2;D2 D2 D2 D2 D2]
    L0 = D1  + im * be * B - im * omega * Ta - be^2 * Vyy + (C .+ im * be * Vyz) * kron(I(5), D)  + (Vzz) * kron(I(5),D2) 
    L1 = im * A - be * Vxy + im *  Vxz * kron(I(5),D)
    L2 = -Vxx
    
    L0 = L0[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    L1 = L1[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    L2 = L2[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
    return L0,L1,L2
end

cof_mat (generic function with 1 method)

In [74]:
A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,-1,Co,D,D2)

(ComplexF64[0.9999994824686826 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.9999979323410284 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; 3.997761898856317e-29 + 0.0im -4.0007860296162177e-29 + 0.0im … 0.008784244630646506 - 15.817097168580098im -0.0013778746714389863 + 0.0im; 1.5204432379096275e-31 + 0.0im -1.5215925168589162e-31 + 0.0im … 6.21774748155261e-5 + 0.0im 0.008431188113647308 - 15.817097168580098im], ComplexF64[0.0 + 285.3598523172633im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 285.35940997283586im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 1.9379638773347288e-29im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 1.9379638773347288e-29im], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 1.3888888866301736 0.0; 0.0 0.0 … 0.0 1.3888888866301736])

In [93]:
L0,L1,L2 = cof_mat(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2)

(ComplexF64[0.9999994824686826 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.9999979323410284 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; 3.997761898856317e-29 + 0.0im -4.0007860296162177e-29 + 0.0im … 0.008784244630646507 - 15.817097168580093im -0.0013778746714389865 + 0.0im; 1.5204432379096275e-31 + 0.0im -1.5215925168589162e-31 + 0.0im … 6.21774748155261e-5 + 0.0im 0.008431188113647308 - 15.817097168580093im], ComplexF64[0.0 + 285.3598523172633im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 285.35940997283586im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 1.9379638773347288e-29im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 1.9379638773347288e-29im], [-0.0 -0.0 … -0.0 -0.0; -0.0 -0.0 … -0.0 -0.0; … ; -0.0 -0.0 … 1.3888888866301736 0.0; -0.0 -0.0 … 0.0 1.3888888866301736])

In [118]:
nep = PEP([L0,L1,L2]); 
eig1,eigvec = iar(nep,σ = 0.3 , neigs = 10 ,maxit = 500,tol=1e-10)
eig1

10-element Vector{ComplexF64}:
  0.2283177411739118 + 0.05216409060076455im
   0.386069447692876 + 0.0003447070955577792im
 0.24572221462875782 + 0.10952523874104671im
 0.33678980774529554 + 0.26578537223774507im
 0.48473402646369834 + 0.2770826859116882im
  0.5610927677137497 + 0.26421689044348673im
  0.3162013722608072 + 0.27590901089521475im
  0.6537809262370726 + 0.22689384482395042im
 0.42096542994750313 + 0.2790147092572307im
  0.3780770221895125 + 0.263124186485235im

In [312]:
function ALST_Operater1(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2)
      if Ro == -1
          Ro = 1
      end

        eye = I(N_cheb + 1)
        Zero = zeros(N_cheb + 1,N_cheb + 1)

        A0_11 = rho .* eye
        A1_11 = im * R * rho .* eye
        A2_11 = Zero

        A0_12 = im * R * be * rho .* eye
        A1_12 = Zero
        A2_12 = Zero

        A0_13 = - R * rho .* (D*rho) .* eye - R * rho.^2 .* D
        A1_13 = Zero
        A2_13 = Zero

        A0_14 = im * R * (be * G .- omega) .* eye + 2 * F .* eye  - (D * rho) .* H .* eye - rho .* H .* D
        A1_14 = im * R * F .* eye
        A2_14 = Zero

        A0_15 = Zero
        A1_15 = Zero
        A2_15 = Zero

        A0_21 = im * R * rho .* (be * G .* eye - omega .* eye) + Ro * (rho .* F .* eye - 2 * rho .* (D*rho) .* H .* eye - rho.^2 .* (D*H) .* eye) + be^2 * T .* eye - D2 * rho .* eye - Ro * rho.^2 .* H .* D - 2 * (D*rho) .* D - rho .* D2
        A1_21 = im * R * rho .* F .* eye
        A2_21 = (lam + 2 * T) .* eye

        A0_22 = -rho .* (2 * Ro * G .+ Co) .* eye
        A1_22 = be * (lam .+ T).*eye
        A2_22 = Zero

        A0_23 = R * rho.^2 .* (D * F) .* eye
        A1_23 = im * ((D*rho).*lam .+ rho .* (D*lam) .+ (1 .+ lam .* rho) .* D) .* eye - im * rho .*(D*T) .*eye
        A2_23 = Zero

        A0_24 = rho .* D2 * F .* eye
        A1_24 = im * R * (gamma * Ma^2)^(-1) * T .* eye
        A2_24 = Zero

        A0_25 = rho .* (D*rho) .* (D*F) .* eye + rho.^2 .*(D*F) .* D
        A1_25 = im * R * (gamma * Ma^2)^(-1) * rho .* eye
        A2_25 = Zero

        A0_31 = rho.*(2*Ro*G .+ Co) .*eye
        A1_31 = be * (lam .+ T).* eye
        A2_31 = Zero

        A0_32 = im * R * rho .* ( be * G .* eye - omega .* eye) + Ro * (rho .* F .* eye - 2 * rho .* (D*rho) .* H .* eye - rho.^2 .* (D*H) .* eye) + be^2 * (lam + 2*T) .* eye - D2 * rho .* eye- Ro * rho.^2 .* H .* D - 2 * rho .* D - rho .* D2
        A1_32 = im * R * rho .* F .* eye
        A2_32 = T .* eye

        A0_33 = R * rho.^2 .* (D*G) .* eye - im * be * rho .* (D*T) .* eye + im * be * ( (D*rho).*lam .+ rho .* (D*lam) .+  (1 .+ lam .* rho) .* D) .* eye
        A1_33 = Zero
        A2_33 = Zero

        A0_34 = F .* (2*Ro * G .+ Co) .* eye+ Ro * rho .* H .* (D*G) .* eye + (gamma * Ma^2)^(-1) .* im * be * R * T .* eye
        A1_34 = Zero
        A2_34 = Zero

        A0_35 = rho .* (D*rho) .* (D*G) .* eye + im * be * R * (gamma * Ma^2)^(-1) * rho .* eye + rho.^2 .* (D * G) .* D
        A1_35 = Zero
        A2_35 = Zero

        A0_41 = Zero
        A1_41 = im* D*rho .* lam .* eye + im * (1 .+ rho.*lam) .* D
        A2_41 = Zero

        A0_42 = im * be * D*rho .* lam .* eye + im * be * (1 .+ rho.*lam) .* D
        A1_42 = Zero
        A2_42 = Zero

        A0_43 = R * rho .* (im * be * G .- im * omega) .* eye  - 2 * Ro * rho .* (D*rho).* H .* eye + be^2 * T .* eye + D2 * (2 * rho .+ lam .* rho.^2) .* eye - Ro * rho.^2 .* H .* D - 4 * (D*rho).*D - 2 * D*lam .* rho.^2 .* D - 4 * (lam) .* rho .*(D*rho).*D - rho .* (2 .+ rho .*lam) .* D2
        A1_43 = im * R * rho .* F .* eye
        A2_43 = T .* eye

        A0_44 =  - R * (gamma * Ma^2)^(-1) * (D*rho) .* T .* eye - R * (gamma * Ma^2)^(-1) * rho .* T .* D
        A1_44 = Zero
        A2_44 = Zero

        A0_45 = -im * be * rho .* (D*G) .* eye - (gamma * Ma^2)^(-1) * R * rho .*(D*rho) .* eye - R *(gamma * Ma^2)^(-1) *rho.^2 .* D
        A1_45 = -im * rho .* (D*F) .* eye
        A2_45 = Zero

        A0_51 = 2 * (gamma-1) * Ma^2 * (rho .* (D2*F) .+ (D*rho).*(D*F)).*eye + 2 * (gamma - 1) * Ma^2 * rho .* (D*F) .* D
        A1_51 = Zero
        A2_51 = Zero

        A0_52 = 2 * (gamma-1) * Ma^2 * (rho .* (D2*G) .+ (D*rho).*(D*G)).*eye + 2 * (gamma - 1) * Ma^2 * rho .* (D*G) .* D
        A1_52 = Zero
        A2_52 = Zero

        A0_53 = -2 * (gamma - 1) * Ma^2 * im * be * (D*G) .* eye + R * rho.^2 .* (D*T) .* eye
        A1_53 = -2 * (gamma - 1) * Ma^2 * im * (D*F) .* eye
        A2_53 = Zero

        A0_54 = (gamma - 1)/(gamma) * (-im * R * T .* (be * G.*eye - omega .* eye) + (D*rho) .* H .* T .* eye + rho .* (D*H) .* T .* eye + rho .* H .* T .* D) + rho .* H .* (D*T) .* eye
        A1_54 = -(gamma - 1)/(gamma) * (im * R * F.* T .* eye)
        A2_54 = Zero

        A0_55 = R * rho .* (im * be * G .* eye - im * omega .* eye) + kappa * be^2 .* eye - (gamma - 1)/(gamma) * (im * be * R .* rho .* G .* eye - im * omega * R .* rho .* eye  - rho .* (D*rho) .* H .* eye - rho.^2 .* (D*H).* eye) -(2 * rho .* (D*rho) .* H .* eye + rho.^2 .* (D*H) .* eye) - (gamma - 1) * Ma^2 * rho.^2 .* ((D*F).^2 .+ (D*G).^2) .* eye  - (1/sigma) * (rho .* (D*rho) .* (D*T) .+(D*rho).*(D*T).+ rho.^2 .* (D2 * T) .+ rho .* (D2 * T) ).* eye -D2*(kappa .* rho.^2) .* eye - rho.^2 .* H .* D  - (1/sigma) * rho.^2 .* (D*T) .* D + (gamma-1)/(gamma) * rho.^2 .* H .* D - 4 * rho .* (D*rho) .* kappa .* D - 2 * rho.^2 .* (D*kappa) .* D - rho.^2 .* kappa .* D2
        A1_55 = im * R * rho .* F .* eye -(gamma-1)/(gamma) * (im * R .* rho .* F) .* eye
        A2_55 = kappa .* eye

        A = [A0_11 A0_12 A0_13 A0_14 A0_15 ; A0_21 A0_22 A0_23 A0_24 A0_25 ; A0_31 A0_32 A0_33 A0_34 A0_35 ; A0_41 A0_42 A0_43 A0_44 A0_45 ; A0_51 A0_52 A0_53 A0_54 A0_55]
        B = [A1_11 A1_12 A1_13 A1_14 A1_15 ; A1_21 A1_22 A1_23 A1_24 A1_25 ; A1_31 A1_32 A1_33 A1_34 A1_35 ; A1_41 A1_42 A1_43 A1_44 A1_45 ; A1_51 A1_52 A1_53 A1_54 A1_55]
        C = [A2_11 A2_12 A2_13 A2_14 A2_15 ; A2_21 A2_22 A2_23 A2_24 A2_25 ; A2_31 A2_32 A2_33 A2_34 A2_35 ; A2_41 A2_42 A2_43 A2_44 A2_45 ; A2_51 A2_52 A2_53 A2_54 A2_55]
        
        A = A[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
        B = B[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
        C = C[setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5)),setdiff(1:end , (1,N_cheb + 1,N_cheb + 2,2N_cheb + 2,2N_cheb + 3,3N_cheb + 3,3N_cheb + 4,4N_cheb + 4,4N_cheb + 5,5N_cheb + 5))]
  
        return A,B,C
end

ALST_Operater1 (generic function with 1 method)

In [ ]:
rho